In [1]:
from statistics import mean
from urllib.parse import quote
import pandas as pd
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


/Users/wangleyuan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
API_KEY = "RGAPI-68e258d1-43b2-46d4-a3e2-b18b2e5195a6"

In [3]:
ROUTING = "americas"
BASE_URL = f"https://{ROUTING}.api.riotgames.com"
HEADERS = {"X-Riot-Token": API_KEY}

MS_PER_DAY = 24 * 60 * 60 * 1000
RETURN_WITHIN_DAYS = 3
FETCH_COUNT = 25
ROLLING_WINDOW = 3


In [4]:
def riot_get(path, params=None):
    r = requests.get(
        BASE_URL + path, headers=HEADERS, params=params or {}, timeout=30
    )
    r.raise_for_status()
    return r.json()


def puuid_from_riot_id(game_name, tag_line):
    gn, tl = quote(game_name, safe=""), quote(tag_line, safe="")
    data = riot_get(f"/riot/account/v1/accounts/by-riot-id/{gn}/{tl}")
    return data["puuid"]


def recent_match_ids(puuid, count):
    return riot_get(
        f"/lol/match/v5/matches/by-puuid/{puuid}/ids",
        params={"count": count},
    )


def match_detail(match_id):
    return riot_get(f"/lol/match/v5/matches/{match_id}")


def participant_row(match, puuid):
    for p in match.get("info", {}).get("participants", []):
        if p.get("puuid") == puuid:
            return p
    return None


def feat_dict_from_participant(p):
    if not p:
        return None
    return {
        "kills": float(p.get("kills") or 0),
        "deaths": float(p.get("deaths") or 0),
        "assists": float(p.get("assists") or 0),
        "goldEarned": float(p.get("goldEarned") or 0),
        "totalDamageDealtToChampions": float(
            p.get("totalDamageDealtToChampions") or 0
        ),
        "totalMinionsKilled": float(p.get("totalMinionsKilled") or 0),
        "visionScore": float(p.get("visionScore") or 0),
        "win": 1.0 if p.get("win") else 0.0,
    }


In [5]:
RIOT_IDS = [
    ("carambolaa", "NA1"),
]

PUUIDS_ONLY = None

In [6]:
def fetch_sorted_matches(puuid):
    mids = recent_match_ids(puuid, FETCH_COUNT)
    matches = [match_detail(mid) for mid in mids]
    matches.sort(key=lambda m: m.get("info", {}).get("gameCreation") or 0)
    return matches


all_matches_by_puuid = {}

if PUUIDS_ONLY:
    for puuid in PUUIDS_ONLY:
        all_matches_by_puuid[puuid] = fetch_sorted_matches(puuid)
else:
    for gn, tl in RIOT_IDS:
        puuid = puuid_from_riot_id(gn, tl)
        all_matches_by_puuid[puuid] = fetch_sorted_matches(puuid)

{puuid: len(ms) for puuid, ms in all_matches_by_puuid.items()}


{'TniKGIQG3GxENjd_9z_VjA9BPwfE5RvxLcozSVT8cQzinQerbgDcOlumAPIaj7MCpyegdxc5XGrPmw': 25}

In [7]:
rows = []

for puuid, matches in all_matches_by_puuid.items():
    n = len(matches)
    if n < 2:
        continue
    thr = RETURN_WITHIN_DAYS * MS_PER_DAY

    for i in range(n - 1):
        m_cur = matches[i]
        m_next = matches[i + 1]
        info_c = m_cur.get("info", {})
        t_cur = info_c.get("gameCreation") or 0
        t_next = (m_next.get("info", {}) or {}).get("gameCreation") or 0
        gap_ms = t_next - t_cur
        label = 1 if gap_ms <= thr else 0

        lo = max(0, i - ROLLING_WINDOW + 1)
        window = matches[lo : i + 1]
        flist = []
        for wm in window:
            fd = feat_dict_from_participant(participant_row(wm, puuid))
            if fd:
                flist.append(fd)
        if not flist:
            continue

        row = {
            "puuid": puuid,
            "matchId": (m_cur.get("metadata") or {}).get("matchId"),
            "retention_proxy": label,
            "roll_kills": mean(x["kills"] for x in flist),
            "roll_deaths": mean(x["deaths"] for x in flist),
            "roll_assists": mean(x["assists"] for x in flist),
            "roll_gold": mean(x["goldEarned"] for x in flist),
            "roll_damage": mean(x["totalDamageDealtToChampions"] for x in flist),
            "roll_cs": mean(x["totalMinionsKilled"] for x in flist),
            "roll_vision": mean(x["visionScore"] for x in flist),
            "roll_winrate": mean(x["win"] for x in flist),
            "n_in_window": len(flist),
            "next_gap_days": gap_ms / MS_PER_DAY,
        }
        if len(window) >= 2:
            gaps_prev = []
            for j in range(len(window) - 1):
                g0 = window[j].get("info", {}).get("gameCreation") or 0
                g1 = window[j + 1].get("info", {}).get("gameCreation") or 0
                gaps_prev.append((g1 - g0) / MS_PER_DAY)
            row["avg_gap_days_in_window"] = mean(gaps_prev)
        else:
            row["avg_gap_days_in_window"] = 0.0

        rows.append(row)

len(rows)


24

In [8]:
df = pd.DataFrame(rows)
print(df.shape)
print(df["retention_proxy"].value_counts())
df.head()


(24, 14)
retention_proxy
1    20
0     4
Name: count, dtype: int64


,puuid,matchId,retention_proxy,roll_kills,roll_deaths,roll_assists,roll_gold,roll_damage,roll_cs,roll_vision,roll_winrate,n_in_window,next_gap_days,avg_gap_days_in_window
0,TniKGIQG3GxENjd_9z_VjA9BPwfE5RvxLcozSVT8cQzinQ...,NA1_5448397525,1,15.000000,5.000000,12.000000,14574.000000,28724.000000,89.000000,0.0,0.000000,1,0.015097,0.000000
1,TniKGIQG3GxENjd_9z_VjA9BPwfE5RvxLcozSVT8cQzinQ...,NA1_5448410159,0,17.500000,8.000000,17.000000,15264.000000,34735.500000,94.500000,0.0,0.500000,2,5.748382,0.015097
2,TniKGIQG3GxENjd_9z_VjA9BPwfE5RvxLcozSVT8cQzinQ...,NA1_5453261119,1,12.666667,8.000000,15.333333,13333.000000,28437.666667,71.666667,0.0,0.333333,3,0.017688,2.881740
3,TniKGIQG3GxENjd_9z_VjA9BPwfE5RvxLcozSVT8cQzinQ...,NA1_5453289255,0,12.666667,10.666667,16.000000,12374.000000,29982.666667,55.333333,0.0,0.333333,3,7.781697,2.883035
4,TniKGIQG3GxENjd_9z_VjA9BPwfE5RvxLcozSVT8cQzinQ...,NA1_5459755644,1,7.333333,9.333333,16.333333,10187.666667,19959.000000,29.000000,22.0,0.333333,3,0.022244,3.899692


In [9]:
feature_cols = [
    "roll_kills",
    "roll_deaths",
    "roll_assists",
    "roll_gold",
    "roll_damage",
    "roll_cs",
    "roll_vision",
    "roll_winrate",
    "n_in_window",
    "avg_gap_days_in_window",
]
target = "retention_proxy"

clean = df.dropna(subset=feature_cols + [target])
X = clean[feature_cols]
y = clean[target].astype(int)

if len(clean) < 15 or y.nunique() < 2:
    print("Too few rows or a single label class. Fetch more matches or try another account.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000)),
        ]
    )
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print("accuracy:", accuracy_score(y_test, pred))
    coefs = pipe.named_steps["clf"].coef_[0]
    print(list(zip(feature_cols, coefs)))


accuracy: 0.8333333333333334
[('roll_kills', np.float64(-0.5694260675367165)), ('roll_deaths', np.float64(-0.8046385207783188)), ('roll_assists', np.float64(0.40254934269390824)), ('roll_gold', np.float64(0.19369965655055918)), ('roll_damage', np.float64(0.2950390644283989)), ('roll_cs', np.float64(-0.3355062127093053)), ('roll_vision', np.float64(0.4402139365981147)), ('roll_winrate', np.float64(-0.04437642804831099)), ('n_in_window', np.float64(0.15178771677817446)), ('avg_gap_days_in_window', np.float64(0.031705084348325634))]
